# MACURA flies a delivery drone
A delivery drone starts about 2.4 m above a landing pad, carrying a package whose weight changes every
flight (the drone knows it). It must come down through gusty wind and land softly. Arriving sooner pays
more, but descending straight down too fast makes the drone lose thrust (a simplified vortex ring state),
and a heavy package leaves less thrust to brake. We compare four reinforcement learning algorithms on the
exact same task, changing only one thing between them: how each uses its learned world model to imagine
extra practice data. MACURA, then MBPO, M2AC and plain SAC, one at a time, then side by side. Runs on a
Kaggle GPU kernel, top to bottom.

## **Get the code**
Clone the project code from GitHub into the kernel. Turn on Internet in the kernel settings and
add a Secret named GITHUB_TOKEN (Add-ons, Secrets) for the private repo.

In [ ]:
import os, sys, subprocess

REPO   = "silvergjeka22/macura-drone"
BRANCH = "cage-env"       # cage + DELIVERY experiments live on this branch (main = ring study)
ROOT   = "/kaggle/working/macura-drone"

# Choose the task BEFORE the config is imported (setup() imports it). Optional overrides:
# os.environ["MACURA_SEEDS"] = "1 2"; os.environ["MACURA_ALGOS"] = "macura mbpo"; os.environ["MACURA_STEPS"] = "2000"
os.environ["MACURA_TASK"] = "delivery"

# GitHub token from a Kaggle Secret (or an env var if you set one)
token = os.environ.get("GITHUB_TOKEN")
if not token:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")

if os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    f"https://{token}@github.com/{REPO}.git", ROOT], check=True)

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

## **Setup**
Install the packages the kernel needs, pick a safe graphics backend, and make the output folders
under /kaggle/working/runs.

In [ ]:
from src.bootstrap import setup
setup(root=ROOT)

In [ ]:
# load every project function into the notebook
%run $ROOT/src/imports.py
set_seed(cfg.SEED)

## **Fresh start**
Wipe old checkpoints, logs, plots and videos so nothing stale is reused.

In [ ]:
import shutil
for d in ("checkpoints", "videos", "plots", "logs"):
    shutil.rmtree(f"{cfg.OUTPUT_ROOT}/{d}", ignore_errors=True)
    os.makedirs(f"{cfg.OUTPUT_ROOT}/{d}", exist_ok=True)
print("cleared old runs:", cfg.OUTPUT_ROOT)

## **Meet the drone**
A clip of the delivery task, flown by a small **hand-written autopilot** (NOT one of the learning
algorithms) that descends slowly: it only shows that the task can be solved under this wind. The windsock
follows the real gusts, the brown box under the drone is the package (bigger = heavier), and the drone
turns **red** whenever it is losing thrust. The four algorithms have to learn the task from scratch.

In [ ]:
path = record_env_video_subprocess(ROOT, f"{cfg.OUTPUT_ROOT}/videos/drone_env.mp4", seconds=10,
                                   policy="autopilot")   # hand-written demo controller, NOT learned
if path:
    display(Video(path, embed=True, width=440))

## **How the environment works**
The same task for all four algorithms, built from a few pieces (figures below):

- **Layout** — the drone spawns about **2.4 m above the pad** (up to 0.5 m to the side) and must come down and settle on it.
- **Package** — every flight carries a package of **0 to 0.2 kg** (up to +42% of the drone's weight). The weight is **in the observation**: the drone knows what it carries.
- **Lift loss** — descending **straight down faster than 1.2 m/s** costs thrust, up to **35% at 2.2 m/s**; flying sideways escapes it. A heavy package leaves less spare thrust to brake, so the safe speed depends on the weight.
- **Wind** — smooth OU gusts plus per-rotor actuator noise, hidden from the agent.
- **Reward** — a pull toward the pad plus a **"settle well"** paid every step the drone is close, slow and upright on the pad, so **arriving sooner pays more**: the temptation to hurry.
- **Difficulty** — under an untrained random policy almost every episode crashes.

*Why this task:* a model trained on the slow descents it has seen is **confidently wrong** about fast,
heavy ones (checked offline: 5-19x larger error there, and 21-27x larger disagreement between its
members). A fixed-horizon method (MBPO) keeps imagining those descents; MACURA stops where its models
disagree. Whether that makes MACURA crash less is exactly what this run measures.

In [ ]:
# how the task is built (the SAME env every algorithm learns on)
plot_env_layout(cfg.CFG, save_path=f"{cfg.OUTPUT_ROOT}/plots/env_layout.png")          # pad + spawn point + package
plot_lift_loss(cfg.CFG, save_path=f"{cfg.OUTPUT_ROOT}/plots/env_lift_loss.png")        # the danger: fast + heavy
plot_wind_process(cfg.CFG, save_path=f"{cfg.OUTPUT_ROOT}/plots/env_wind.png")          # wind gusts + actuator noise
plot_reward_landscape(cfg.CFG, save_path=f"{cfg.OUTPUT_ROOT}/plots/env_reward.png")    # what the reward rewards
plot_env_difficulty(cfg.CFG, save_path=f"{cfg.OUTPUT_ROOT}/plots/env_difficulty.png"); # how hard the raw env is

## **How the comparison works**
All four algorithms share the same policy learner (SAC), the same learned world model, the same
exploration and the same evaluation seeds. Only the rollout differs:
- SAC uses no model (it learns only from real flights).
- MBPO imagines fixed length rollouts.
- M2AC imagines fixed length rollouts, then drops the least trustworthy ones.
- MACURA imagines rollouts but stops each one as soon as the model becomes unsure.

Because only the rollout differs, any difference in the results comes from that choice alone.

## **MACURA**
MACURA trusts the model only where the model trusts itself. Step by step:
1. Imagine short rollouts from real states, using the learned world model.
2. At each step, measure how much the ensemble of models disagree (its uncertainty).
3. Keep imagining while the uncertainty stays low, and stop that rollout the moment it gets
   high. Safe regions get long rollouts, risky regions get short ones.
4. Train the SAC policy on this clean imagined data plus a little real data.

This is the method we want to test.

In [ ]:
macura_runs = train_algo("macura", cfg.CFG, cfg.OUTPUT_ROOT)

In [ ]:
plot_sample_efficiency(macura_runs, f"{cfg.OUTPUT_ROOT}/plots/macura_return.png")
plot_failure_rate(macura_runs, f"{cfg.OUTPUT_ROOT}/plots/macura_crash.png");
plot_rollout_depth(best_run(macura_runs), f"{cfg.OUTPUT_ROOT}/plots/macura_rollout_depth.png");
# where MACURA stops trusting its model: fast vs slow descents
plot_trust_by_zone(macura_runs, f"{cfg.OUTPUT_ROOT}/plots/macura_trust_by_descent.png", split="descent");

Test: fly the best MACURA policy and look at its numbers and a short clip.

In [ ]:
b = best_run(macura_runs)
print("MACURA best seed", b["seed"],
      "-> return", round(b["final_eval"]["eval_return"], 1),
      "| crash", round(b["final_eval"]["eval_failure_rate"], 2),
      "| reach", round(b["final_eval"]["eval_success_rate"], 2))
# ~1-min highlight reel: many episodes, best ones (landings first) stitched with per-clip captions
macura_video = record_policy_video(b["checkpoint"], cfg.CFG, f"{cfg.OUTPUT_ROOT}/videos/macura.mp4",
                                   seconds=60, label="MACURA")
if macura_video:
    display(Video(macura_video, embed=True, width=440))

## **MBPO**
MBPO is the baseline MACURA builds on. Step by step:
1. Imagine rollouts from real states, using the world model.
2. Use a fixed rollout length from a schedule (it does not look at uncertainty).
3. Train SAC on all of that imagined data plus a little real data.

Because the length is fixed, in risky regions MBPO keeps imagining past the point where the
model is reliable, which can feed the policy misleading data.

In [ ]:
mbpo_runs = train_algo("mbpo", cfg.CFG, cfg.OUTPUT_ROOT)

In [ ]:
plot_sample_efficiency(mbpo_runs, f"{cfg.OUTPUT_ROOT}/plots/mbpo_return.png")
plot_failure_rate(mbpo_runs, f"{cfg.OUTPUT_ROOT}/plots/mbpo_crash.png");

Test: fly the best MBPO policy and look at its numbers and a short clip.

In [ ]:
b = best_run(mbpo_runs)
print("MBPO best seed", b["seed"],
      "-> return", round(b["final_eval"]["eval_return"], 1),
      "| crash", round(b["final_eval"]["eval_failure_rate"], 2),
      "| reach", round(b["final_eval"]["eval_success_rate"], 2))
mbpo_video = record_policy_video(b["checkpoint"], cfg.CFG, f"{cfg.OUTPUT_ROOT}/videos/mbpo.mp4",
                                 seconds=60, label="MBPO")
if mbpo_video:
    display(Video(mbpo_video, embed=True, width=440))

## **M2AC**
M2AC also imagines fixed length rollouts, then filters them. Step by step:
1. Imagine fixed length rollouts.
2. Score each imagined step by how much the models disagree.
3. Keep only the most trustworthy fraction and drop the rest.
4. Train SAC on what is left plus a little real data.

It reacts to uncertainty by filtering after the fact, instead of stopping early.

In [ ]:
m2ac_runs = train_algo("m2ac", cfg.CFG, cfg.OUTPUT_ROOT)

In [ ]:
plot_sample_efficiency(m2ac_runs, f"{cfg.OUTPUT_ROOT}/plots/m2ac_return.png")
plot_failure_rate(m2ac_runs, f"{cfg.OUTPUT_ROOT}/plots/m2ac_crash.png");

Test: fly the best M2AC policy and look at its numbers and a short clip.

In [ ]:
b = best_run(m2ac_runs)
print("M2AC best seed", b["seed"],
      "-> return", round(b["final_eval"]["eval_return"], 1),
      "| crash", round(b["final_eval"]["eval_failure_rate"], 2),
      "| reach", round(b["final_eval"]["eval_success_rate"], 2))
m2ac_video = record_policy_video(b["checkpoint"], cfg.CFG, f"{cfg.OUTPUT_ROOT}/videos/m2ac.mp4",
                                 seconds=60, label="M2AC")
if m2ac_video:
    display(Video(m2ac_video, embed=True, width=440))

## **SAC**
SAC is the model free baseline. It has no world model and never imagines anything: it learns
only from the real flights it collects. It is the reference for how much the world model helps
the other three.

In [ ]:
sac_runs = train_algo("sac", cfg.CFG, cfg.OUTPUT_ROOT)

In [ ]:
plot_sample_efficiency(sac_runs, f"{cfg.OUTPUT_ROOT}/plots/sac_return.png")
plot_failure_rate(sac_runs, f"{cfg.OUTPUT_ROOT}/plots/sac_crash.png");

Test: fly the best SAC policy and look at its numbers and a short clip.

In [ ]:
b = best_run(sac_runs)
print("SAC best seed", b["seed"],
      "-> return", round(b["final_eval"]["eval_return"], 1),
      "| crash", round(b["final_eval"]["eval_failure_rate"], 2),
      "| reach", round(b["final_eval"]["eval_success_rate"], 2))
sac_video = record_policy_video(b["checkpoint"], cfg.CFG, f"{cfg.OUTPUT_ROOT}/videos/sac.mp4",
                                seconds=60, label="SAC")
if sac_video:
    display(Video(sac_video, embed=True, width=440))

## **All algorithms compared**
Now all four on the same axes. Read the average return over training, the crash rate and the stability
first, then the final return. The final return re-tests each method's single best checkpoint on **fresh
scenarios** (not the ones it was picked on).

Then the **why**: how much of the imagined data each method trained on was data its own models
**disagreed about** (above MACURA's trust threshold), how much of it was **fast descents** (where the lift
loss makes the model wrong), and whether the crashes happen with **heavy packages**.

In [ ]:
runs = macura_runs + mbpo_runs + m2ac_runs + sac_runs
plot_overview(runs, f"{cfg.OUTPUT_ROOT}/plots/overview.png")
plot_iqm_efficiency(runs, f"{cfg.OUTPUT_ROOT}/plots/iqm.png")
plot_crash_rate_ci(runs, f"{cfg.OUTPUT_ROOT}/plots/crash_ci.png")
plot_steps_to_threshold(runs, save_path=f"{cfg.OUTPUT_ROOT}/plots/steps_to_threshold.png")
plot_summary_table(runs, f"{cfg.OUTPUT_ROOT}/plots/summary.png")
# WHY: how much untrustworthy imagined data each method trains on (MACURA filters it out)
plot_untrusted_data(runs, f"{cfg.OUTPUT_ROOT}/plots/untrusted_data.png")
# WHY (delivery): imagined FAST descents each method trains on, and crashes by package weight
plot_fast_data(runs, f"{cfg.OUTPUT_ROOT}/plots/fast_descent_data.png")
plot_crash_by_payload(runs, f"{cfg.OUTPUT_ROOT}/plots/crash_by_payload.png");

## **All videos side by side**
The four best policies in one clip, each labelled, so you can watch the difference directly.

In [ ]:
clips = {"MACURA": macura_video, "MBPO": mbpo_video, "M2AC": m2ac_video, "SAC": sac_video}
clips = {k: v for k, v in clips.items() if v}
if clips:
    reel = stitch_videos_grid(clips, f"{cfg.OUTPUT_ROOT}/videos/reel.mp4", cols=2)
    display(Video(reel, embed=True, width=560))

## **The race: MACURA vs MBPO on the same flights**
Both best policies fly the **same four fresh scenarios** (same package, same start, same gusts), side by
side, with a running scoreboard of deliveries and crashes. Trail dots are green while the drone has full
thrust and red while it is losing it.

In [ ]:
race = record_race_video(cfg.CFG, {"MACURA": best_run(macura_runs)["checkpoint"],
                                   "MBPO": best_run(mbpo_runs)["checkpoint"]},
                          f"{cfg.OUTPUT_ROOT}/videos/race_macura_mbpo.mp4", seeds=(1000, 1001, 1002, 1003))
if race:
    display(Video(race, embed=True, width=720))

## **Save**
Everything is under /kaggle/working/runs (checkpoints, logs, plots, videos) and is saved as the
kernel output. Download it, then watch a policy live on your Mac with:

`mjpython run_live_mac.py --compare out/runs/checkpoints`